# Module 3: Datasets and Experiments

> Self-directed module, ~30 min.

Picking up from Module 2: the agent runs and produces traces. This module covers two ways to evaluate it offline, without waiting for production traffic:

- **Part A: Traditional evals** — a fixed dataset of examples, run functions, and evaluators. The agent runs in-process; each example's input maps to a known output shape.
- **Part B: Environment evals** — the agent runs inside a real sandboxed container instead, and a verifier grades what it actually did to the filesystem, after the fact.

The investment research agent is a supervisor delegating to two subagents (`research-advisor`, `research-digest`), with `research-advisor` able to interrupt mid-run to ask a clarifying question. Part A's evaluation splits into two layers:

1. **End-to-end** (Parts 2-4) — run the full supervisor and score the final answer, the first delegation decision, and the full trajectory of subagent hops.
2. **Subcomponent** (Part 5) — run `research-advisor` and `research-digest` standalone and score each in isolation, since a component regression can hide behind a passing end-to-end score.

Part 2 also shows the interrupt/resume mechanic: one example deliberately triggers `research-advisor`'s clarifying question, and the run function resumes on the same thread so the pre- and post-interrupt turns land as one connected run in LangSmith.

Online evaluations (Module 4) reuse the same judge pattern on every new trace as it lands, instead of running on a fixed dataset.


## Setup

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import re
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.models import model
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.types import Command
from collections import Counter
from typing import Any

# Standalone subagent builders + the report corpus, used by Part 5's
# subcomponent evals and by the citation-coverage check in Part 2.
from agents.investment_research_agent import (
    build_research_advisor_agent,
    build_research_digest_agent,
    _REPORTS,
)


## Part 1. Dataset

A LangSmith dataset is a versioned collection of examples, each with `inputs` and (optionally) `outputs`. The outputs are reference data for evaluators — a success rubric for LLM judges, or expected tool sequences for trajectory evaluation. The dataset is browsable and editable in the UI after creation; the SDK is for programmatic creation and scripting.

Five examples cover the supervisor's main paths: single-hop lookup, retrieval + digest handoff, multi-tool retrieval within a subagent, the clarifying-question interrupt, and the not-in-corpus edge case. Each example carries three reference fields so the same dataset drives Parts 2-4:

- `reference_answer` — a success rubric (not verbatim expected text) for the LLM judge
- `expected_first_tool` — the first tool call we expect the agent to make
- `trajectory` — the full expected sequence of tool calls, recorded as `task:<subagent_name>` for each delegation

The interrupt example (D) additionally carries a `resume_answer` **input** — the canned answer the run function supplies once `research-advisor` asks its clarifying question.

In [ ]:
examples = [
    {
        "inputs": {"query": "What's our latest research on Apple's earnings?"},
        "outputs": {
            "reference_answer": (
                "Identifies Apple's most recent earnings_review report(s) (Q2 FY26: EPS $1.62 vs. "
                "$1.54 consensus, revenue $96.1B, Services growth 14% YoY), citing the report's "
                "source_url. Should not present the June price-target update as an earnings result."
            ),
            "expected_first_tool": "task:research-advisor",
            "trajectory": ["task:research-advisor"],
        },
    },
    {
        "inputs": {"query": "Digest our Costco earnings review into a two-sentence summary for a client email."},
        "outputs": {
            "reference_answer": (
                "A short (roughly two-sentence), client-facing digest of Costco's Q3 FY26 earnings "
                "review — comp sales +6.8%, membership renewal 92.9% — citing the source_url, with "
                "no internal jargon like 'report_type'."
            ),
            "expected_first_tool": "task:research-advisor",
            "trajectory": ["task:research-advisor", "task:research-digest"],
        },
    },
    {
        "inputs": {"query": "Give me our internal view on Apple plus what's happening with the stock today."},
        "outputs": {
            "reference_answer": (
                "Includes both an internal citation on Apple (any AAPL report's source_url) and "
                "external market/web context on Apple's current stock action, clearly distinguishing "
                "internal analysis from external market data."
            ),
            "expected_first_tool": "task:research-advisor",
            "trajectory": ["task:research-advisor"],
        },
    },
    {
        "inputs": {
            "query": "What have we published recently?",
            "resume_answer": "I mean Apple -- anything from this year is fine.",
        },
        "outputs": {
            "reference_answer": (
                "Does not guess a ticker, date range, or report type up front. Asks a clarifying "
                "question first; after being told 'Apple, anything from this year', returns Apple "
                "report(s) from 2026 with citations."
            ),
            "expected_first_tool": "task:research-advisor",
            "trajectory": ["task:research-advisor"],
        },
    },
    {
        "inputs": {"query": "Any updates on Novatera?"},
        "outputs": {
            "reference_answer": (
                "States plainly that Novatera isn't a name covered by internal research, without "
                "fabricating a ticker, report, or source_url."
            ),
            "expected_first_tool": "task:research-advisor",
            "trajectory": ["task:research-advisor"],
        },
    },
]

dataset_name = "investment-research-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


### Verify in LangSmith

Open `dataset.url` and confirm all examples are present. The UI is where examples get edited, added, or split into train/test groups — once iteration starts, expect to mostly work with this dataset from the UI rather than re-running this cell.

<img src="../images/dataset.png" alt="LangSmith dataset detail view with five examples" style="width: auto; max-height: 360px; border-radius: 8px;">

## Part 2. Final-Response Evaluation

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

End-to-end evaluation: run the supervisor on each example's input, capture the final response, score it two ways — an LLM judge against the success rubric (**final-answer relevance and completeness**), and a code check that every cited source is real and not cited twice (**citation coverage and deduplication**). Both scores land in the same experiment since they run against the same output.

The run function invokes the agent and, if `research-advisor` interrupts with a clarifying question, resumes on the **same thread_id** with the example's `resume_answer` before reading the final response.

In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Target function: run the agent, resuming past any clarifying-question interrupt."""
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    if result.get("__interrupt__"):
        # Two separate agent.invoke() calls, two separate root traces in LangSmith --
        # linked only by sharing this thread_id. See the next cell for why that's fine here.
        answer = inputs.get("resume_answer", "Use your best judgment.")
        result = agent.invoke(Command(resume=answer), config=config)

    return {"response": result["messages"][-1].content}


### Why this interrupt doesn't complicate the eval — and where it would

`agent.invoke()` runs until `research-advisor` interrupts, then returns with `__interrupt__` set; the second `agent.invoke(Command(resume=...))` call resumes it. LangSmith traces each call as its own root run, so this produces two traces sharing one `thread_id`, not one continuous trace.

That split doesn't matter here — `client.evaluate()` only sees what `run_agent_final()` returns, and the function makes both calls itself before handing back one final answer. It does matter in production, where the interrupt and the resume are genuinely separate events with no such wrapper. That's what Module 4's thread-level evaluators are for.

The correctness evaluator uses the **LLM-as-judge** pattern: `TypedDict` with `Annotated` field metadata gives the structured-output binding both type safety and inline documentation the judge sees as part of the schema. The prompt explicitly distinguishes "meets the rubric" from "matches the rubric verbatim."

The citation evaluator is a plain code check — no model call — since "is this URL one we actually have" and "is it cited more than once" are both exact, verifiable facts about the response text.

In [ ]:
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the rubric."""
    score: Annotated[bool, ..., "True if the response meets the success rubric"]
    reasoning: Annotated[str, ..., "One short sentence of reasoning"]

correctness_judge_prompt = """You are grading an investment research assistant.

You will see the analyst's request, the assistant's final response, and a RUBRIC describing success.

The rubric is success criteria, not the expected response text. Mark score=True if the response
demonstrates the criteria were met -- even if worded differently from the rubric. Mark score=False
only if the response clearly missed the task, hallucinated a ticker/report/figure, or refused
without good reason.

Give one short sentence of reasoning either way.
"""

judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"Analyst request: {inputs['query']}\n\n"
            f"Assistant response: {outputs.get('response', '')}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "final_answer_relevance_completeness", "score": int(grade["score"]), "comment": grade["reasoning"]}


_KNOWN_SOURCE_URLS = {r["source_url"] for r in _REPORTS}

def citation_evaluator(outputs, reference_outputs):
    """Every cited URL must be one of our reports, and none should be cited twice."""
    cited = re.findall(r"https://research\.internal/reports/\S+", outputs.get("response", ""))
    cited = [url.rstrip(").,") for url in cited]
    unknown = [url for url in cited if url not in _KNOWN_SOURCE_URLS]
    duplicates = len(cited) - len(set(cited))
    score = int(not unknown and duplicates == 0)
    return {
        "key": "citation_coverage_dedup",
        "score": score,
        "comment": f"unknown_urls={unknown}, duplicate_citations={duplicates}",
    }


Expect each example to take roughly 10-30 seconds depending on how many tool calls the agent makes, plus one extra turn for the interrupt example. With `max_concurrency=2` and five examples, the full experiment runs in a couple of minutes.

In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator, citation_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


Open `dataset.url` and switch to the **Experiments** tab. The experiment just created appears at the top; clicking it opens the per-example view with the agent's response and both scores (`final_answer_relevance_completeness`, `citation_coverage_dedup`) column-by-column. Open the interrupt example's row and follow its trace — it shows the two invoke calls sharing one thread.

## Part 3. Single-Step Evaluation

<img src="../images/single-step.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Final-response evaluation answers "did the output meet the bar." Single-step evaluation answers "where did things go right or wrong inside the agent." Every one of our examples expects the supervisor to delegate to `research-advisor` first — it never holds a data-fetch tool of its own — so this check confirms the supervisor never skips delegation and calls something else (or nothing) first.

Why bother with this when final-response already covers correctness? Final-response is too coarse to diagnose drift. If a regression lands and the final-response score drops, single-step tells you whether the supervisor is now delegating to the wrong place first (planning regression) or delegating correctly and producing a bad answer downstream (generation regression).

In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names in order; `task` calls become `task:<subagent_name>`."""
    names = []
    for msg in messages:
        for tc in getattr(msg, "tool_calls", None) or []:
            if tc["name"] == "task":
                names.append(f"task:{tc['args'].get('subagent_type', 'unknown')}")
            else:
                names.append(tc["name"])
    return names

def run_agent_first_tool(inputs: dict) -> dict:
    """Single-step target: capture only the first tool the agent called."""
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    tool_calls = extract_tool_calls(result["messages"])
    return {"first_tool": tool_calls[0] if tool_calls else None}

def first_tool_match(outputs, reference_outputs):
    """Exact match between the agent's first tool call and the expected one."""
    return {
        "key": "first_tool_match",
        "score": int(outputs.get("first_tool") == reference_outputs["expected_first_tool"]),
    }


Run it — same dataset as Part 2, different run function and evaluator.

In [ ]:
results = client.evaluate(
    run_agent_first_tool,
    data=dataset_name,
    evaluators=[first_tool_match],
    experiment_prefix="single-step",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


In the Experiments tab, the `single-step` experiment shows up alongside `final-response`. The per-example column shows the agent's first tool vs the expected one — a `0` is a single-step regression even when final-response might still pass (the agent recovered downstream).

## Part 4. Trajectory Evaluation

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Trajectory evaluation scores the full sequence of subagent delegations — this is where **tool-call correctness** (query decomposition, output handoff, orchestration) and **tool-call accuracy** (do the calls made match what was expected) get scored together. Three metrics:

- **Exact match** — does the trajectory equal the reference, in order?
- **Extra steps** — how many delegations did the agent make that weren't in the reference (e.g. an unnecessary `research-digest` hop)?
- **Missing steps** — how many expected delegations did the agent skip (e.g. answering without ever calling `research-digest` when a digest was asked for)?

Extra and missing surface drift even when exact match is too strict. Trajectory complements single-step: single-step catches the first divergence; trajectory catches *all* of them, including whether a two-hop handoff (retrieve, then digest) actually happened.

In [ ]:
def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs.get("trajectory", []) == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs.get("trajectory", [])) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs.get("trajectory", []))
    return {"key": "missing_steps", "score": sum(missing.values())}


Run it — one experiment, all three trajectory metrics attached.

In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


## Part 5. Subcomponent Evaluation

End-to-end scores can hide a regression in one subagent behind a compensating success (or failure) somewhere else in the graph. The standard way to test one subagent in isolation is to invoke it as its own callable and run `client.evaluate` against it, scoped to that subagent -- the same pattern used for the supervisor above, just narrower.

`agents/investment_research_agent.py` exposes `build_research_advisor_agent()` and `build_research_digest_agent()` for exactly this — each builds the subagent standalone, using the same tools and system prompt as its entry in the supervisor's `subagents=[...]` list.

### 5a. research-advisor

Two things matter for `research-advisor` on its own: does it retrieve the *right* reports (**retrieval relevance**), and does it correctly interrupt for missing information rather than guessing (**clarification trigger correctness**)? Two well-specified queries and one underspecified one cover both.

In [ ]:
advisor_examples = [
    {
        "inputs": {"query": "Show me our most recent price target update on Costco."},
        "outputs": {"expected_source_urls": ["https://research.internal/reports/r8"], "should_interrupt": False},
    },
    {
        "inputs": {"query": "Pull our Exxon sector note on refining margins."},
        "outputs": {"expected_source_urls": ["https://research.internal/reports/r6"], "should_interrupt": False},
    },
    {
        "inputs": {"query": "What have we published recently?"},
        "outputs": {"expected_source_urls": [], "should_interrupt": True},
    },
]

advisor_dataset_name = "investment-research-evals-advisor"

if client.has_dataset(dataset_name=advisor_dataset_name):
    existing = client.read_dataset(dataset_name=advisor_dataset_name)
    client.delete_dataset(dataset_id=existing.id)

advisor_dataset = client.create_dataset(advisor_dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in advisor_examples],
    outputs=[e["outputs"] for e in advisor_examples],
    dataset_id=advisor_dataset.id,
)
print(f"Created dataset '{advisor_dataset_name}' with {len(advisor_examples)} examples")


`research-advisor` invoked directly, skipping the supervisor entirely — its own dataset, scoped to what it alone is responsible for.

In [ ]:
def run_research_advisor(inputs: dict) -> dict:
    """Invoke research-advisor standalone and report whether it interrupted."""
    advisor = build_research_advisor_agent()
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = advisor.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    interrupted = bool(result.get("__interrupt__"))
    response = "" if interrupted else result["messages"][-1].content
    return {"response": response, "interrupted": interrupted}


Two code checks, not LLM judges — retrieval either found the right report or it didn't, and the interrupt either fired or it didn't. Both are exact, verifiable facts.

In [ ]:
def retrieval_relevance(outputs, reference_outputs):
    expected_urls = reference_outputs.get("expected_source_urls") or []
    if not expected_urls:
        return {"key": "retrieval_relevance", "score": None, "comment": "not applicable -- clarifying-question example"}
    hit = any(url in outputs.get("response", "") for url in expected_urls)
    return {"key": "retrieval_relevance", "score": int(hit)}


def clarification_trigger_correctness(outputs, reference_outputs):
    return {
        "key": "clarification_trigger_correctness",
        "score": int(outputs.get("interrupted") == reference_outputs["should_interrupt"]),
    }


Run it — one experiment, both advisor evaluators attached.

In [ ]:
results = client.evaluate(
    run_research_advisor,
    data=advisor_dataset_name,
    evaluators=[retrieval_relevance, clarification_trigger_correctness],
    experiment_prefix="research-advisor",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


### 5b. research-digest

`research-digest` never searches — it only summarizes report text it's handed. Its examples pass report content directly as input, mirroring what the supervisor hands it after a retrieval step. Two LLM judges score **summary faithfulness** (no claims beyond what's in the source) and **summary completeness** (the rubric's key points are covered).

A second dataset, scoped to `research-digest` alone: report text in, a digest out.

In [ ]:
digest_examples = [
    {
        "inputs": {"report_text": _REPORTS[6]["full_text"]},  # Costco Q3 FY26 earnings review (r7)
        "outputs": {
            "key_points_rubric": (
                "Must mention comparable sales growth (~6.8%) and the membership renewal rate "
                "(~92.9%). Must not invent a price target, since none is in the source."
            ),
        },
    },
    {
        "inputs": {"report_text": _REPORTS[4]["full_text"]},  # Exxon initiation of coverage (r5)
        "outputs": {
            "key_points_rubric": (
                "Must cover the Equal Weight thesis (upstream cost discipline and Permian growth "
                "offset by normalized refining margins) and at least one named risk (crude price "
                "volatility, refining margin compression, or Pioneer integration execution)."
            ),
        },
    },
]

digest_dataset_name = "investment-research-evals-digest"

if client.has_dataset(dataset_name=digest_dataset_name):
    existing = client.read_dataset(dataset_name=digest_dataset_name)
    client.delete_dataset(dataset_id=existing.id)

digest_dataset = client.create_dataset(digest_dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in digest_examples],
    outputs=[e["outputs"] for e in digest_examples],
    dataset_id=digest_dataset.id,
)
print(f"Created dataset '{digest_dataset_name}' with {len(digest_examples)} examples")


`research-digest` has no tools — the target function hands it report text directly, the same way the supervisor would after a retrieval step.

In [ ]:
def run_research_digest(inputs: dict) -> dict:
    digest = build_research_digest_agent()
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = digest.invoke(
        {"messages": [{"role": "user", "content": f"Summarize this report:\n\n{inputs['report_text']}"}]},
        config=config,
    )
    return {"response": result["messages"][-1].content}


Two LLM judges, same pattern as Part 2's `CorrectnessGrade` — one scores faithfulness to the source, the other scores coverage of the rubric's key points.

In [ ]:
class FaithfulnessGrade(TypedDict):
    """Score whether the digest stayed faithful to the source report."""
    score: Annotated[bool, ..., "True if the digest contains no claim absent from the source report"]
    reasoning: Annotated[str, ..., "One short sentence of reasoning"]

faithfulness_judge_prompt = """You are grading a research-digest agent for fabricated claims.

Compare the digest against the source report it was given. Mark score=True if every claim in the
digest is supported by the source. Mark score=False if the digest states a figure, rating, or
fact that isn't in the source report.
"""

faithfulness_judge = model.with_structured_output(FaithfulnessGrade)

def summary_faithfulness(inputs, outputs):
    grade = faithfulness_judge.invoke([
        SystemMessage(content=faithfulness_judge_prompt),
        HumanMessage(content=f"Source report: {inputs['report_text']}\n\nDigest: {outputs.get('response', '')}"),
    ])
    return {"key": "summary_faithfulness", "score": int(grade["score"]), "comment": grade["reasoning"]}


class CompletenessGrade(TypedDict):
    """Score whether the digest covered the rubric's key points."""
    score: Annotated[bool, ..., "True if the digest covers the key points in the rubric"]
    reasoning: Annotated[str, ..., "One short sentence of reasoning"]

completeness_judge_prompt = """You are grading a research-digest agent for completeness.

The rubric lists the key points a good digest should cover. Mark score=True if the digest covers
them (in any wording), even briefly. Mark score=False if it omits a required point.
"""

completeness_judge = model.with_structured_output(CompletenessGrade)

def summary_completeness(inputs, outputs, reference_outputs):
    grade = completeness_judge.invoke([
        SystemMessage(content=completeness_judge_prompt),
        HumanMessage(content=f"Digest: {outputs.get('response', '')}\n\nKey points rubric: {reference_outputs['key_points_rubric']}"),
    ])
    return {"key": "summary_completeness", "score": int(grade["score"]), "comment": grade["reasoning"]}


Run it — one experiment, both digest evaluators attached.

In [ ]:
results = client.evaluate(
    run_research_digest,
    data=digest_dataset_name,
    evaluators=[summary_faithfulness, summary_completeness],
    experiment_prefix="research-digest",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


### Reviewing experiments

Each dataset's **Experiments** tab is the central place to inspect, compare, and iterate. `investment-research-evals` carries `final-response`, `single-step`, and `trajectory`; `investment-research-evals-advisor` and `investment-research-evals-digest` each carry one subcomponent experiment. Every row shows summary metrics at a glance.

<img src="../images/experiments_page.png" alt="LangSmith Experiments tab listing multiple experiment runs against the dataset" style="width: auto; max-height: 360px; border-radius: 8px;">

### Compare experiments

On a dataset's **Experiments** tab, select two or more experiments and use **Compare** to put them side by side. The per-example diff highlights where one experiment scored differently from another — the fastest way to see whether a change to an agent (or one subagent's prompt) moved the needle, or just moved a few examples around.

<img src="../images/experiment_comparison.png" alt="Experiment comparison view" style="width: auto; max-height: 360px; border-radius: 8px;">

### Set an experiment as baseline

Hovering over an experiment row reveals a **Set as baseline** action. Once set, the baseline pins to the top of the Experiments tab and every other experiment shows performance deltas against it across each metric column — a fast way to see whether the most recent run is an improvement, a regression, or a wash.

Useful whenever you're holding most variables constant and varying one thing (a prompt tweak, a model swap, a tool change) to measure its effect in isolation — including a change scoped to just one subagent.

<img src="../images/set_baseline_button.png" alt="Setting an experiment as baseline via hover menu in LangSmith" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

<img src="../images/set_baseline.png" alt="Experiments list with baseline pinned and performance deltas surfaced per metric column" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

## Recap

| Layer | Pattern | Scores | Reference field(s) |
|---|---|---|---|
| End-to-end | Final-response | `final_answer_relevance_completeness`, `citation_coverage_dedup` | `reference_answer` |
| End-to-end | Single-step | First delegation decision | `expected_first_tool` |
| End-to-end | Trajectory | `tool_call_correctness` / `tool_call_accuracy` (exact + extra + missing) | `trajectory` |
| Subcomponent | research-advisor | `retrieval_relevance`, `clarification_trigger_correctness` | `expected_source_urls`, `should_interrupt` |
| Subcomponent | research-digest | `summary_faithfulness`, `summary_completeness` | `key_points_rubric` |

That's all 8 evaluation criteria from the spec, split across the layer they actually diagnose. Run all of them on every iteration — an end-to-end regression tells you *that* something broke; the subcomponent experiments tell you *which* subagent.

Part B takes two of these same criteria -- faithfulness and completeness -- and scores them a different way: on a file the agent actually wrote inside a sandboxed environment, rather than a value it returned.


---
# Part B: Environment Evals

Part A's evaluations all share a shape: a fixed input, a known-good output shape, a run function that calls the agent in-process and returns a value. That covers most of what needs checking, but not a specific, real problem: confirming something an agent *wrote* is actually correct means re-deriving it independently -- and if what it wrote includes code, re-deriving it means executing code the agent authored. That's not something you want to do inside the same process running your evaluation suite.

[Harbor](https://www.harborframework.com/) gives you a disposable, isolated place to do that safely: the agent runs inside a real sandboxed environment, and a **verifier** grades what it actually produced there afterward -- including re-running code the agent wrote, without that code touching anything else.

**Vocabulary**, one line each:
- **task** — an instruction, an environment, and a verifier, bundled together
- **environment** — the sandboxed container the agent runs in
- **verifier** — the script that grades what's on disk once the agent stops
- **trial** — one agent attempt at one task
- **job** — a batch of trials

One task, on disk:

```
earnings-first-take/              <- the task
├── deep-agent/
│   ├── agent.py                  <- the agent Harbor loads (--agent langgraph)
│   └── langgraph.json
├── task.toml                     <- config: timeouts, network policy, secrets
├── instruction.md                <- the prompt handed to the agent
├── environment/
│   ├── Dockerfile                <- builds the sandbox
│   ├── press_release.md          <- seeded input
│   └── consensus.txt             <- seeded input
├── solution/
│   └── solve.sh                  <- oracle: sanity-checks the verifier
└── tests/
    ├── test.sh                   <- entry point Harbor runs once the agent stops
    └── verify.py                 <- deletes the prose, re-runs analysis.py, grades
```

### Three ways this plugs into LangSmith

- **Sandboxes** (`--env langsmith`) — the container runs on LangSmith-hosted infrastructure instead of local Docker.
- **Deep Agents** (`--agent langgraph`) — Harbor loads a `langgraph.json` graph and calls it once per trial, the same `create_deep_agent()` shape used throughout this repo.
- **Evaluations** (`--plugin langsmith`) — the job syncs to LangSmith as a dataset + experiment, the same objects Part A produces.

### Why this matters beyond evaluation

The verifier in B.2 does something a return-value check never could: it deletes the agent's written note and re-runs the agent's *own* analysis script from a clean state, then checks the two agree. Executing code the agent authored is exactly the kind of thing that belongs in a disposable, isolated environment, not your own evaluation process.

The reward this produces is also a training signal, not only a pass/fail for a demo -- the same task/environment/verifier scaffolding that grades an agent today is what feeds post-training (fine-tune or RL against your own graded trajectories) and harness optimization (iterate on prompts, tools, or architecture against that same reward).

This section is illustrative: one task, five independent attempts, one agent.

## The trap

Consensus is quoted on an **adjusted** basis; the press release headlines **GAAP**. Dividing the two without adjusting for the basis mismatch inverts the conclusion:

| | |
|---|---|
| GAAP net income | $412M |
| Litigation charge (pre-tax) | $340M |
| Tax rate | 21% |
| Diluted shares | 380M |
| Consensus adjusted EPS | $1.71 |

- Naive: `412 / 380 = 1.08` → looks like a 37% miss
- Correct: `(412 + 340 × 0.79) / 380 = 1.79` → actually a 4.7% beat

## B.1 The agent under test

A fresh, minimal agent — not the supervisor/subagent architecture from Parts 1-5. Its system prompt only sets its role; the actual requirements (what to read, what to write, the reproducibility rule) live in `instruction.md` instead, which is also what would let this same task run against a different agent harness with no code changes. It lives at `environment_evals/earnings-first-take/deep-agent/agent.py`, separate from `agents/`, because Harbor loads it fresh inside a container on every trial.

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import subprocess
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langsmith import uuid7

from utils.models import model
from utils.harbor_utils import seed_workspace

RESEARCH_ASSOCIATE_PROMPT = (
    "You are a research associate. Read the source files before answering. "
    "Use plain relative file paths (e.g. `analysis.py`, `report/take.md`) -- "
    "not absolute paths -- for every file you read or write."
)

def build_local_agent():
    # The real container agent (deep-agent/agent.py) pins root_dir="/app" -- Harbor's
    # langgraph runner doesn't necessarily execute from the task's working directory.
    # Locally we control the cwd ourselves, so root_dir="." + chdir is enough.
    #
    # virtual_mode=False for the same reason as the container agent: with
    # virtual_mode=True, an absolute path the agent sees from `execute`'s real
    # `pwd` gets re-rooted under root_dir instead of used as-is, landing writes
    # one directory too deep. See deep-agent/agent.py for the full explanation.
    return create_deep_agent(
        model=model,
        system_prompt=RESEARCH_ASSOCIATE_PROMPT,
        backend=LocalShellBackend(root_dir=".", virtual_mode=False),
    )


Seed a scratch workspace from the task's own `environment/` -- the same two files its Dockerfile copies into the container -- and run the agent against it directly. No Harbor yet.

In [ ]:
task_dir = project_root / "environment_evals" / "earnings-first-take"
workspace = seed_workspace(task_dir / "environment", project_root / ".dev-workspace")

os.chdir(workspace)
agent = build_local_agent()
result = agent.invoke(
    {"messages": [{"role": "user", "content": (task_dir / "instruction.md").read_text()}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)

take_path = workspace / "report" / "take.md"
analysis_path = workspace / "analysis.py"
print("=== report/take.md ===")
print(take_path.read_text() if take_path.exists() else "(not written)")
print("\n=== analysis.py ===")
print(analysis_path.read_text() if analysis_path.exists() else "(not written)")


## B.2 The verifier

The verifier doesn't grade the prose — it deletes it, re-runs the agent's own `analysis.py` fresh, and checks that the script's output, the now-deleted prose, and the correct figure all agree. Three checks, one combined score:

- `analysis_reproduces` — the script runs cleanly from a clean state
- `eps_correct` — it actually computes the adjusted figure, not the naive one
- `prose_ties_to_code` — the deleted note agreed with what the code says

No LLM judge here — "does the math check out" is an exact fact, not a judgment call, so a judge would only add noise to a check that's better done deterministically.

Note that `verify()` **deletes** `report/take.md` -- that's the point, not a side effect. Re-running this cell on its own will therefore score `prose_ties_to_code: 0`, since the prose is already gone; re-run the B.1 cell first to regenerate it.


In [ ]:
import json

def verify(take_path: Path, analysis_cwd: Path) -> dict:
    """Delete the agent's prose, re-run its own script fresh, check they agree."""
    prose = take_path.read_text() if take_path.is_file() else ""
    take_path.unlink(missing_ok=True)

    try:
        p = subprocess.run(["python3", "analysis.py"], cwd=analysis_cwd,
                            capture_output=True, text=True, timeout=60)
        rc, out = p.returncode, p.stdout
    except Exception:
        rc, out = 1, ""

    reward = {
        "analysis_reproduces": float(rc == 0),
        "eps_correct": float("1.79" in out),
        "prose_ties_to_code": float("1.79" in prose),
    }
    reward["reward"] = round(sum(reward.values()) / len(reward), 4)
    return reward

print(json.dumps(verify(take_path, workspace), indent=2))


## B.3 Packaging it as a Harbor task

The function above is exactly what runs inside the container — packaged as `tests/verify.py`, called by `tests/test.sh`, writing `/logs/verifier/reward.json`. The container can't import this repo, so `verify.py` duplicates the logic rather than importing it (the same self-contained-file convention used everywhere else in this repo). The cells below read the real files straight off disk, the same way Module 6 shows `langgraph.json` and `AGENTS.md` — nothing here is a separate hand-copied version that could drift from what Harbor actually runs.

`task.toml` — task metadata and phase timeouts:

In [ ]:
task_toml_path = task_dir / "task.toml"
with open(task_toml_path) as f:
    print("task.toml:")
    print(f.read())


`environment/Dockerfile` — seeds the two inputs the agent needs, and the empty folder it writes to:

In [ ]:
dockerfile_path = task_dir / "environment" / "Dockerfile"
with open(dockerfile_path) as f:
    print("Dockerfile:")
    print(f.read())


`instruction.md` — the prompt handed to the agent, including the reproducibility requirement:

In [ ]:
instruction_path = task_dir / "instruction.md"
with open(instruction_path) as f:
    print("instruction.md:")
    print(f.read())


`solution/solve.sh` — the oracle: a hand-written correct answer, used to sanity-check the verifier itself before spending any tokens on a real model (B.4).

In [ ]:
solve_path = task_dir / "solution" / "solve.sh"
with open(solve_path) as f:
    print("solution/solve.sh:")
    print(f.read())


`tests/verify.py` — B.2's function, unimported, writing the reward:

In [ ]:
verify_path = task_dir / "tests" / "verify.py"
with open(verify_path) as f:
    print("tests/verify.py:")
    print(f.read())


`tests/test.sh` — the entry point Harbor actually runs, with a fallback reward if the verifier itself errors:

In [ ]:
test_sh_path = task_dir / "tests" / "test.sh"
with open(test_sh_path) as f:
    print("tests/test.sh:")
    print(f.read())


The whole task, in one tree:

In [ ]:
from utils.harbor_utils import print_tree

print_tree(task_dir)


## B.4 Run it

The `harbor` CLI is already available -- it's a regular project dependency (`harbor[langsmith]`, same as `langgraph-cli` is for Module 6), installed by Module 0's `uv sync`. Nothing extra to install here.

One thing to confirm before running:

- `--env langsmith` requires sandboxes to be enabled on your LangSmith workspace. If a run below fails on environment startup, confirm that with your workspace admin before debugging anything else.
- `LANGSMITH_API_KEY` and `LANGSMITH_ENDPOINT` -- already set from Module 0's setup.
- The task's agent (`deep-agent/agent.py`) routes through the LangSmith Gateway, which also requires workspace access. Swap its `init_chat_model` call for a direct provider if you don't have it.

Before spending any tokens on a real model, sanity-check the verifier against the oracle -- a hand-written correct answer, not an LLM. Expect `"reward": 1.0` in the printed output.

In [ ]:
!cd "{project_root}" && harbor run -p environment_evals/earnings-first-take -a oracle --env langsmith -y


Now the real agent — `-k 5` runs five independent attempts at the same task (not `-n`, which only bounds concurrency), so we can see whether the trap actually fires. All five landing the same way either means the trap is too easy to notice or too obscure to matter; a mix is the interesting result.

In [ ]:
# Runs on a LangSmith-hosted sandbox (--env langsmith) rather than local Docker, and
# syncs the job to LangSmith as a dataset + experiment (--plugin langsmith).
!cd "{project_root}" && harbor run -p environment_evals/earnings-first-take \
  --agent langgraph --ak project_path=./environment_evals/earnings-first-take/deep-agent --ak graph=deep_agent \
  --env langsmith \
  -k 5 -n 5 \
  --plugin langsmith --pk dataset_name=earnings-first-take-eval --pk experiment_name=earnings-first-take \
  -y


Confirm the sync landed -- same `dataset.url` pattern Part A uses:

In [ ]:
from langsmith import Client

ls_client = Client()
synced_dataset = ls_client.read_dataset(dataset_name="earnings-first-take-eval")
print(f"Dataset: {synced_dataset.url}")


The experiment the plugin created is a normal LangSmith project -- same `list_runs` + `list_feedback` idiom Module 4 uses to pull back online-eval feedback, just applied to a Harbor job's trials instead of live traces. Five trials, all the same task -- look for a mix of high and low `reward` values, not all-1.0 or all-0.0.

In [ ]:
experiment_project = ls_client.read_project(project_name="earnings-first-take")
print(f"Experiment: {experiment_project.url}")

trials = list(ls_client.list_runs(project_name="earnings-first-take", is_root=True))
print(f"{len(trials)} trial(s) synced\n")

for trial in trials:
    feedback = list(ls_client.list_feedback(run_ids=[trial.id]))
    scores = {f.key: f.score for f in feedback}
    print(f"  {trial.name:20s} {scores}")


## Recap

| | Part A: Traditional evals | Part B: Environment evals |
|---|---|---|
| Execution | In-process, `agent.invoke()` | Sandboxed container (LangSmith-hosted) |
| Scored on | A returned value | Files the agent actually wrote, including re-executing its own code |
| Verifier | Run function + evaluator, in Python | `tests/verify.py`, packaged with the task |
| Sanity check | -- | The oracle (`solution/solve.sh`), before spending any tokens |
| LangSmith object | Dataset + experiment, via `client.evaluate()` | Dataset + experiment, via `--plugin langsmith` |

Same underlying question, two different execution models. Part A is cheaper to iterate on and covers most of what needs checking; Part B is what to reach for once verifying an agent's output means re-deriving it independently -- especially when that means executing code the agent wrote. For the full version of this pattern -- larger task datasets, model/prompt matrices, CI-gated regression checks -- see [harborframework.com](https://www.harborframework.com/).

Next: `04_online_evals.ipynb` — the same judge pattern Part A used, applied automatically to every new trace as it lands.
